In [ ]:
import tdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
from sklearn.decomposition import PCA
import seaborn as sb
import sys
import trf
from scipy.stats import pearsonr
import matplotlib.colors as mcolors
import functools
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal


# import my own modules
from QC_utils import * # my QC functions
sys.path.insert(0, str(Path.cwd().parent))
from analysis_utils import *  # imported from the parent directory via sys.path

import logging
from pathlib import Path

# Perform sample QC using tdb and custom QC methods
This notebook uses the functions found in `QC_utils.py` to:
1. Parse and separate the sequences corresponding to disease associated regions that are found in variation clusters and construct updated tdb style tables for the separated sequences
1. Read the tdb database containing all genotyped regions
1. Append the updated tables to the existing tdb database
1. Perform sample QC with the combined database to identify alleles and loci that qualify for analysis

## Parse disease associated variation clusters

In [ ]:
daTR_loci = [922520,1033259,3192251,3240174,3631877,4053467]
daTR_locus_map = pd.DataFrame({
    "LocusID": [922520, 1033259, 3192251, 3240174, 3631877, 4053467],
    "abbreviation": ["DM2", "HD", "SCA8", "SCA27B", "SCA31", "SCA36"]
})

# load tdb with added MS field
curr = tdb.load_tdb(
    "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/all_HPRC_MS.tdb",
    lfilters=[("LocusID", "in", daTR_loci)]
)
# load original tdb with purity field (AP)
curr_AP= tdb.load_tdb("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/all_HPRC_100_with_AP.tdb", lfilters=[( 'LocusID', 'in', daTR_loci)])

# load repeatmasker and add the mapping from AP to the MS allele numbers
repeatmasker = tdb_qc.load_repeatmasker_data("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/repeatmasker_filter/all_HPRC_100_with_AP_alleles_all.out")
tdb_mappings = tdb_MS_qc.get_tdb_mappings(curr_AP, curr)
del curr_AP
# add the mapping for allele numbers to the repeatmasker data
repeatmasker = pd.merge(repeatmasker, tdb_mappings, left_on=["LocusID", "allele_number"], right_on = ["LocusID","allele_number_AP"],how="left")
curr_meta = pd.merge(left = curr["allele"],right=repeatmasker[["LocusID","allele_number_AP","allele_number_MS","query_start","query_end","subfamily","family","element_length"]], left_on=["LocusID","allele_number"],right_on=["LocusID","allele_number_MS"], how="left")
curr_meta = tdb_qc.calculate_repeatmasker_span(curr_meta)
curr_meta.drop_duplicates(subset=["LocusID","allele_number"], inplace=True)
curr_meta.head()

In [ ]:
# read in input data to get motifs that correspond to the spans
# only load the data for the regions defined in Variation cluster bed file
vc_only = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/variation_clusters_only.bed", sep="\t", header=None)
vc_only.columns = ["chrom", "start", "end", "vc_id"]

vc_only["vc_TRIDs"] = vc_only.vc_id.apply(tdb_MS_qc.parse_vcid)
vc_only["vc_motifs"] = vc_only.vc_id.apply(tdb_MS_qc.parse_vcid_motifs)

vc_only["num_TRIDs"] = vc_only.vc_TRIDs.apply(len)
vc_only["num_motifs"] = vc_only.vc_motifs.apply(len)

vc_only["motif_periods"] = vc_only.vc_motifs.apply(tdb_MS_qc.get_motif_periods)
vc_only["max_period"] = vc_only.motif_periods.apply(max)
vc_only["min_period"] = vc_only.motif_periods.apply(min)
vc_only["mean_period"] = vc_only.motif_periods.apply(np.mean)

print(vc_only.num_TRIDs.value_counts())
print(vc_only.num_motifs.value_counts())

# combine with tdb
vc_only["tdb_mapping"] = vc_only.chrom + ":" + (vc_only.start -1).astype(str)+ "-" + vc_only.end.astype(str)
curr["locus"]["tdb_mapping"] = curr["locus"].chrom + ":" + (curr["locus"].start).astype(str)+ "-" + curr["locus"].end.astype(str)
# add the tdb_mapping to the allele table
curr_meta = pd.merge(left = curr_meta, right=curr["locus"][["LocusID","tdb_mapping"]], on=["LocusID"], how="left")
# add vc_motifs column to the allele table
curr_meta = pd.merge(left = curr_meta, right=vc_only[["tdb_mapping","vc_motifs"]], on=["tdb_mapping"], how="left")
curr_meta.head()

In [ ]:
# get motif spans
all_allele_data = tdb_MS_qc.add_purity_MS(curr)
curr_meta = pd.merge(left = curr_meta, right=all_allele_data[["LocusID","allele_number","motif_span"]], on=["LocusID","allele_number"], how="left")
del all_allele_data
# get percent spanned
curr_meta["motif_span"] = curr_meta["motif_span"].fillna(".")
# curr_meta["percent_spanned"] = curr_meta.apply(lambda x: get_percent_spanned(x["allele_length"], x["motif_span"]), axis=1)
# get the split sequences
split_seqs = curr_meta.apply(lambda x: tdb_MS_qc.split_sequence_MS(x["LocusID"], x["allele_number"],x["allele_length"], x["sequence"], x["motif_span"],x["vc_motifs"]), axis=1)
split_seqs = pd.concat(split_seqs.tolist())
split_seqs = pd.merge(split_seqs, daTR_locus_map, on=["LocusID"], how="left")
split_seqs.head()

In [ ]:
curr_copy = {}
curr_copy["locus"] = curr["locus"].copy()
curr_copy["allele"] = curr["allele"].copy()
curr_copy["sample"] = {}
for sample in curr["sample"].keys():
    curr_copy["sample"][sample] = curr["sample"][sample].copy()
_stdout = sys.stdout
try:
    with open("/dev/null", "w") as _null:
        sys.stdout = _null
        vc_daTR_tdb = tdb_MS_qc.update_tables(split_seqs, curr_copy)
finally:
    sys.stdout = _stdout

# plot the HD locus spans
plot_df = vc_daTR_tdb["allele"][
    (vc_daTR_tdb["allele"]["original_LocusID"] == 1033259)
    & (vc_daTR_tdb["allele"]["span_number"].isin([1, 2]))
].copy()

plot_df["span"] = plot_df["span_number"].map({1: "CAG (span 1)", 2: "CCG (span 2)"})

sb.violinplot(
    data=plot_df,
    x="span",
    y="allele_length",
    inner="box",
    palette=["red", "blue"]
)
plt.title("Example: separating the HTT locus into pathogenic CAG and flanking CCG repeat spans")
plt.xlabel("Repeat")
plt.ylabel("Allele length")
plt.tight_layout()

In [ ]:
# save to a pickle file for QC
import pickle
updated_pkl = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_vc_tables_final.pkl"
with open(updated_pkl, 'wb') as f:
    pickle.dump(vc_daTR_tdb, f)

## Integrate updated database and perform sample QC

In [ ]:
# load the locusid coordinate mapping from the locus table of the tdb (previously saved)
full_tdb_bed = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/variation_clusters_and_isolated_TRs_v1.hg38.TRGT.motif_columns.LocusID.AP_tdb.bed"
# load the coordinates for the isolated STRs to add as our initial filters

#TODO use the original isolated STR file instead
isolated_strs = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/isolated_TRs_v1.hg38.TRGT.no_overlapping_regions.STRs.bed", sep="\t", header=None)
# isolated_strs = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/isolated_only_STRs.bed", sep="\t", header=None)
isolated_strs.columns = ["chrom", "start", "end", "TRID", "motif", "period"]
# start coordinates are off by 1
isolated_strs["start"] = isolated_strs["start"] - 1
# merge to get the locus IDs for the isolated STRs
full_tdb = pd.read_csv(full_tdb_bed, sep="\t", header=None)
full_tdb.columns = ["chrom", "start", "end", "TRID", "motif", "period", "LocusID"]
isolated_strs_with_locus = pd.merge(isolated_strs, full_tdb, on=["chrom", "start", "end", "TRID", "motif", "period"], how="left")
# save with the start reset to its original value
isolated_strs_with_locus["start"] = isolated_strs_with_locus["start"] + 1
# isolated_strs_with_locus.to_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/isolated_only_STRs_with_locus.bed", sep="\t", index=False)
isolated_strs_with_locus.to_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/isolated_only_STRs_with_locus_original.bed", sep="\t", index=False)


isolated_strs_with_locus

In [ ]:
updated_pkl = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_vc_tables_final.pkl"
period_path = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/repeatmasker_filter/all_HPRC_100_MS_periods_revised_input_match_final.tsvt"
repeatmasker_path = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/repeatmasker_filter/all_HPRC_100_MS_alleles_all_final.out"
period_locusids = '/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/merged_starting_MEIs/MEI_and_non_MEI_STRs.period.locusid.txt'
# run QC

# get original locusIDs for what we are adding so we load those too
new_ids = vc_daTR_tdb["locus"].original_LocusID.unique().astype(int).tolist() # this is so they are loaded
updated_ids = vc_daTR_tdb["locus"].LocusID.unique().astype(int).tolist() # not all in tdb, but all need to be ignored since new sequences
not_included = [101068, 511206, 1324732, 2374646, 208930, 2615070, 2909520, 3073063, 3236837, 3265856, 3560092, 3895180, 4022110, 517099, 4206663, 4222385, 828822, 936991, 1092625, 1263986, 1602219, 1650746, 1832861, 1875734, 2238420, 2339401, 2473933, 4320484, 4376762, 4482278, 4492755, 4493523] # daTRs not in VCs but not isolated -- likely defined twice in the catalog
to_exclude = updated_ids + not_included # not_included do not have percent_match stats yet so ignore them in the allele_QC step -- still subject to minimum spanning reads, new ids are not cause they are loaded in update
# duplicate ids for this database (these break most of their query functions so we need to remove them) 
duplicate_ids_2 = [146007, 146008, 153932, 153933, 202833, 202834, 202873, 202874, 228825, 228826, 236718, 236719, 332520, 332521, 342649, 342650, 374478, 374479, 530443, 530444, 561319, 561320, 596205, 596206, 638757, 638758, 692283, 692284, 865335, 865336, 1009965, 1009966, 1026115, 1026116, 1085002, 
                   1085003, 1167612, 1167613, 1179203, 1179204, 1235715, 1235716, 1310101, 1310102, 1332693,1332694, 1421990, 1421991, 1438240, 1438241, 1480591, 1480592, 1485693, 1485694, 1536775, 1536776, 1660311, 1660312, 1664132, 1664133, 1706476, 1706477, 1925633, 1925634, 2028244, 2028245, 2216718, 
                   2216719, 2287396, 2287397, 2333679, 2333680, 2387552, 2387553, 2686568, 2686569, 2842433, 2842434, 2921241, 2921242, 2991647, 2991648, 3086363, 3086364, 3109212, 3109213, 3226021, 3226022, 3250881, 3250882, 3336357, 3336358, 3374196, 3374197, 3489847, 3489848, 3512952, 3512953, 3765517, 
                   3765518, 3787698, 3787699, 3845421, 3845422, 3912217, 3912218, 3945087, 3945088, 3951187, 3951188, 4193802, 4193803, 4271411, 4271412, 4277194, 4277195, 4286652, 4286653]

isolated_locusIDs = isolated_strs_with_locus.LocusID.unique().tolist()
all_str_locusids = new_ids + not_included + isolated_locusIDs # also load the ids that are not in VCs but were not previously included

initial_lfilters = [("LocusID", "in", all_str_locusids),( 'LocusID', 'not in', duplicate_ids_2)]
data_path = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/all_HPRC_MS.tdb"
cleaned_data, final_afilters, final_lfilters = tdb_MS_qc.run_all_QC(data_path,period_loci_path= period_locusids, period_stats_path=period_path, repeatmasker_path=repeatmasker_path,initial_lfilters = initial_lfilters, min_allele_support=100, save_filters=True, 
                                                                    min_spanning_reads= 10 ,filter_path = "./daTR_update_10spanning", update_tdb = updated_pkl, exclude_allele_QC=to_exclude)

In [ ]:
# save the filtered and updated allele data for downstream analysis
cleaned_data["allele"].to_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_update_10spanning.csv", index=False)

In [ ]:
cleaned_data['allele'][cleaned_data['allele']['span_number']==2]

# Update the labelling for the newly added VC disease associated reagions

In [ ]:
def update_labels(current_labels, new_labels, new_tdb):
    ''' 
    This is needed to deal with the fact that we want the original and new locusids in the labels for
    loci that were split
    '''
    # check if the update has the period for the split motifs
    if "period" not in new_tdb["locus"].columns and "motif" in new_tdb["locus"].columns:
        # if not, add it
        new_tdb["locus"]["period"] = new_tdb["locus"]["motif"].str.len()
    assert "period" in new_tdb["locus"].columns
    # add LocusIDs and original LocusIDs to the new labels
    assert "original_LocusID" in new_tdb["locus"].columns
    # adjust the start
    new_labels["start_tdb"] = new_labels["start_str"] - 1

    # get locusid based on start_tdb and end_str merged on locus table start and end
    if "motif" in new_tdb["locus"].columns:
        new_labels = pd.merge(new_labels,new_tdb["locus"][["chrom", "start", "end", "period","original_LocusID","LocusID","motif"]], left_on=["chrom", "start_tdb", "end_str"], right_on=["chrom", "start", "end"]).reset_index(drop=True)
        new_labels["str_motif"] = new_labels["motif_y"]
        new_labels.drop(columns=["motif_y", "motif_x"], inplace=True)
    else:
        new_labels = pd.merge(new_labels,new_tdb["locus"][["chrom", "start", "end", "period","original_LocusID","LocusID"]], left_on=["chrom", "start_tdb", "end_str"], right_on=["chrom", "start", "end"]).reset_index(drop=True)
    new_labels["str_period"] = new_labels["period"]
    new_labels.drop(columns=["period","start","end"], inplace=True)

    # update columns to match the current labels produced in analysis_utils
    new_labels['TRID'] = new_labels['str_id'].str.split('ID=', expand=True)[1].str.split(';', expand=True)[0]
    new_labels.rename(columns={"start_str":"start", "end_str":"end"}, inplace=True) # Changed since start off by one relative to tdb -- needs to lineup -- may have done the opposite previously?
    
    if "original_LocusID" not in current_labels.columns:
        current_labels["original_LocusID"] = current_labels["LocusID"]
    else:
        # fill in missing original_LocusIDs with the current LocusID
        current_labels["original_LocusID"] = current_labels["original_LocusID"].fillna(current_labels["LocusID"])
    current_labels.reset_index(drop=True, inplace=True)
    new_labels.reset_index(drop=True, inplace=True)
    print(current_labels.index.is_unique)

    all_labels = pd.concat([current_labels[["LocusID","original_LocusID","TRID", "chrom","start","end","str_period", "str_motif",]], 
                                    new_labels[["LocusID","original_LocusID","TRID", "chrom","start","end","str_period", "str_motif"]]], ignore_index=True)
    
    
    return all_labels

In [ ]:
daTR_labels = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/daTR_lables_LocusIDs.bed", sep="\t")
# load the isolated STRs with locusIDs
curr_labels = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/isolated_only_STRs_with_locus_original.bed", sep="\t")
# update the TRID column
curr_labels['TRID'] = curr_labels['TRID'].str.split('ID=', expand=True)[1].str.split(';', expand=True)[0]
# change motif column name to str_motif
curr_labels = curr_labels.rename(columns={"motif": "str_motif", "period": "str_period"})

print(curr_labels.shape[0])
# get locusid based on start_tdb and end_str merged on locus table start and end
updated_labels = update_labels(curr_labels, daTR_labels, cleaned_data)
print(updated_labels.shape[0])
# updated_labels = update_labels(updated_labels, daTR_non_mei_labels, cleaned_data)
# # add motifs to the labels where they are not using the TRID
updated_labels["str_motif"] = updated_labels["str_motif"].fillna(updated_labels["TRID"].str.split("-", expand=True)[3])
updated_labels["str_period"] = updated_labels["str_period"].fillna(updated_labels["str_motif"].str.len())

In [ ]:
# save off
updated_labels.to_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/updated_labels.bed", sep="\t", index=False)

## Exclude problematic regions
1. Blacklist
1. Satellites
1. Centromeres

In [ ]:
def _preprocess_satellite_metadata(labels, metadata_paths):
    """
    Preprocess the satellite metadata
    Arguments:
        metadata_paths: dictionary of paths to the satellite metadata including repeatmasker 
                        satellites from hg38 and repeatmasker outputs from all alleles and allele_table
    Returns:
        DataFrame: DataFrame with the satellite metadata preprocessed

    NOTE this could be separated or at least the calculations could be helper functions
    """
    # Load the satellite metadata from the provided paths
    allele_table = pd.read_csv(metadata_paths["allele_table"])
    all_alleles_repeatmasker = pd.read_csv(metadata_paths["repeatmasker_all_alleles"], sep="\t", header=None)

    all_alleles_repeatmasker = all_alleles_repeatmasker.dropna()
    all_alleles_repeatmasker.columns = ["LocusID:allele","query_start","query_end","subfamily","family"]
    # split the LocusID:allele column
    all_alleles_repeatmasker["LocusID"] = all_alleles_repeatmasker["LocusID:allele"].str.split(":", expand=True)[0].astype(int)
    all_alleles_repeatmasker["allele_number"] = all_alleles_repeatmasker["LocusID:allele"].str.split(":", expand=True)[1].astype(int)
    allele_satellites = all_alleles_repeatmasker[all_alleles_repeatmasker["family"].str.contains("Satellite")]
    allele_satellites["satellite_allele"] = True

    # use the allele_table to get the number of total alleles per locus with a satellite
    allele_table = allele_table.merge(allele_satellites[["LocusID","allele_number","satellite_allele"]], on=["LocusID","allele_number"], how='left')
    allele_table["satellite_allele"] = allele_table["satellite_allele"].fillna(False)

    # calculate the percent of alleles labelled as satellite per locus 
    allele_table["percent_satellite_allele"] = allele_table.groupby("LocusID").satellite_allele.transform("sum")/allele_table.groupby("LocusID").allele_number.transform("count")
    # also add a column indicating at least one allele is labelled as satellite
    allele_table["satellite_allele_labelled"] = allele_table.groupby("LocusID").satellite_allele.transform("sum") > 0
    
    #TODO I changed the allele_table back to include all loci because when we merge back with the data we don't want nan values
    # reduce allele_table to unique on LocusID
    allele_table_locus = allele_table.drop_duplicates(subset = "LocusID")

    # load hg38 repeatmasker data
    hg38_repeatmasker = pd.read_csv(metadata_paths["repeatmasker_hg38"], sep="\t", header=None)
    hg38_repeatmasker.columns = ["chrom","start","end","repeat_subfamily","repeat_family"]
    hg38_repeatmasker["satellite_hg38"] = True

    # get the coordinates of our full input from the mapping table
    TRID_LocusID_Coords_map = labels[["chrom","start","end","LocusID","TRID"]]
    # get the pyranges object for this data by renaming the chrom, start, and end columns to expected input
    hg38_repeatmasker_gr = pr.PyRanges(hg38_repeatmasker.rename(columns={"chrom":"Chromosome","start":"Start","end":"End"}))
    TRID_LocusID_Coords_map_gr = pr.PyRanges(TRID_LocusID_Coords_map.rename(columns={"chrom":"Chromosome","start":"Start","end":"End"}))
    overlaps = TRID_LocusID_Coords_map_gr.join(hg38_repeatmasker_gr, report_overlap= True, suffix="_repeatmasker")
    overlaps_df = overlaps.df
    overlaps_df["satellite_hg38"] = True
    overlaps_df = overlaps_df.rename(columns={"Chromosome":"chrom","Start":"start","End":"end"})
    # merge back to TRID_LocusID_Coords_map
    TRID_LocusID_Coords_map = pd.merge(TRID_LocusID_Coords_map, overlaps_df[["chrom","start","end","satellite_hg38"]], on=["chrom","start","end"], how="left")
    TRID_LocusID_Coords_map["satellite_hg38"] = TRID_LocusID_Coords_map["satellite_hg38"].fillna(False)

    # merge the allele_table locus with the TRID_LocusID_Coords_map on LocusID
    TRID_LocusID_Coords_map = pd.merge(TRID_LocusID_Coords_map, allele_table_locus[["LocusID","percent_satellite_allele","satellite_allele_labelled"]], on="LocusID", how="left")
    TRID_LocusID_Coords_map["percent_satellite_allele"] = TRID_LocusID_Coords_map["percent_satellite_allele"].fillna(0)
    TRID_LocusID_Coords_map["satellite_allele_labelled"] = TRID_LocusID_Coords_map["satellite_allele_labelled"].fillna(False)
    
    return TRID_LocusID_Coords_map.drop_duplicates(subset = "LocusID")

def _get_exclusions(labels,exclusion_paths):
    """
    Get the exclusions from the exclusion paths
    Arguments:
        exclusion_paths: dictionary of paths to the exclusion files (ie centromeres, telomeres, blacklist, satellites ect.)
    Returns:
        DataFrame: DataFrame with the exclusions
    """
    exclusions = []
    for exclusion_type, exclusion_path in exclusion_paths.items():
        exclusion = pd.read_csv(exclusion_path, sep="\t", header=None)[[0,1,2]] # only get coordinate columns
        exclusion.columns = ["Chromosome","Start","End"]
        exclusion["exclusion_type"] = exclusion_type
        exclusions.append(exclusion)
    exclusions_df = pd.concat(exclusions)
    # merge with our mapping table
    TRID_LocusID_Coords_map = labels[["chrom","start","end","LocusID","TRID"]]
    TRID_LocusID_Coords_map_gr = pr.PyRanges(TRID_LocusID_Coords_map.rename(columns={"chrom":"Chromosome","start":"Start","end":"End"}))
    exclusions_gr = pr.PyRanges(exclusions_df)
    overlaps = TRID_LocusID_Coords_map_gr.join(exclusions_gr, report_overlap= True, suffix="_exclusion")
    overlaps_df = overlaps.df
    # merge back to TRID_LocusID_Coords_map
    overlaps_df = overlaps_df.rename(columns={"Chromosome":"chrom","Start":"start","End":"end"})
    TRID_LocusID_Coords_map = pd.merge(TRID_LocusID_Coords_map, overlaps_df[["chrom","start","end","exclusion_type"]], on=["chrom","start","end"], how="left")
    TRID_LocusID_Coords_map["exclusion_type"] = TRID_LocusID_Coords_map["exclusion_type"].fillna("keep")
    return TRID_LocusID_Coords_map.drop_duplicates(subset = "LocusID")

In [ ]:
# read the labels
updated_labels = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/sorted_VCs_overlaps/updated_labels.bed", sep="\t")
print(updated_labels.shape[0]) 

In [ ]:
# paths for regions to exclude
repeatmasker_hg38 = "/data/projects/nanopore/RepeatExpansion/coordinates/repeatmasker_hg38.Satellite.bed"
repeatmasker_all_alleles = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/repeatmasker_filter/all_HPRC_100_with_AP_alleles_all.out"
allele_table = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_update_10spanning.csv"
genomic_annotations = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/genome_annotations/gencode.v47.basic.annotation.master_regions.bed"
genomic_annotations = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/genome_annotations/gencode.v47.basic.annotation.master_regions_lncRNA.bed"

exclusion_paths = {}
exclusion_paths["centromeres"]= "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/genome_annotations/centromeres.bed"
exclusion_paths["blacklist"] = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/genome_annotations/blacklist.bed"
exclusion_paths["satlellites"] = "/data/projects/nanopore/RepeatExpansion/coordinates/repeatmasker_hg38.Satellite.bed"

In [ ]:
# add labels
satellite_labels = _preprocess_satellite_metadata(updated_labels, metadata_paths={"repeatmasker_hg38":repeatmasker_hg38, "repeatmasker_all_alleles":repeatmasker_all_alleles,"allele_table":allele_table})
satellites = satellite_labels[(satellite_labels.satellite_hg38) | (satellite_labels.percent_satellite_allele > 0)]
print("there are ", updated_labels.LocusID.isin(satellites.LocusID).sum(), "satellite loci")

# exclusions from hg38 repeatmasker and other regions
repeatmasker_exclusions = _get_exclusions(updated_labels, exclusion_paths=exclusion_paths)

repeatmasker_exclusions.exclusion_type.value_counts()

In [ ]:
# add exclusion_type to the labels and filter
# merge updated_labels with satellite_labels to get satellite_allele_labelled column
updated_labels = updated_labels.merge(satellite_labels[["LocusID","satellite_allele_labelled"]], on = "LocusID", how="left")
# add the exclusion_type column from repeatmasker_exclusions
updated_labels = updated_labels.merge(repeatmasker_exclusions[["LocusID","exclusion_type"]], on = "LocusID", how="left")
updated_labels["exclusion_type"] = np.where(updated_labels["satellite_allele_labelled"], "satellite_allele_labelled", updated_labels["exclusion_type"] )
updated_labels.exclusion_type.value_counts()

In [ ]:
updated_labels.exclusion_type.value_counts()

In [ ]:
# save out the final filtered updated_labels
updated_labels = updated_labels[updated_labels.exclusion_type == "keep"]
updated_labels.to_csv("final_updated_labels.tsv", sep="\t", index=False)

# also save a bed file version as chrom start end locusid
updated_labels_bed = updated_labels[['chrom', 'start', 'end', 'LocusID']]
updated_labels_bed.to_csv("final_updated_labels.bed", sep="\t", index=False, header=False)
